# TMR AI Text Detector — Colab API Server
**Model:** `Oxidane/tmr-ai-text-detector` / `adnanchy083/tmr-ai-text-detector-bucket` (RoBERTa trained on RAID dataset)

### Quick Setup:
1. (Optional) Set runtime to **GPU**: *Runtime -> Change runtime type -> T4 GPU*
2. Run **Cell 1** to install dependencies
3. Run **Cell 2** to download the model and launch the public API server
4. Copy the **`Running on public URL: https://xxxx.gradio.live`** link
5. In your ATS Agent `.env` file, set:
   ```bash
   COLAB_DETECTOR_URL=https://xxxx.gradio.live
   ```
6. Restart your Flask server (`py app.py`) and use the **AI Lab** tab in your browser!

> **Keep this Colab tab open** while using the AI detector.

In [ ]:
# Cell 1: Install required dependencies
!pip install -q transformers torch gradio accelerate
print('Dependencies installed successfully!')

In [ ]:
# Cell 2: Load TMR Model and launch Gradio API Server
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Candidate model paths
MODELS_TO_TRY = [
    'Oxidane/tmr-ai-text-detector',
    'adnanchy083/tmr-ai-text-detector-bucket'
]

tokenizer = None
model = None
loaded_model_name = ''

for model_id in MODELS_TO_TRY:
    try:
        print(f'Loading model: {model_id} ...')
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSequenceClassification.from_pretrained(model_id)
        loaded_model_name = model_id
        break
    except Exception as e:
        print(f'Note on {model_id}: {e}')

if model is None:
    raise RuntimeError('Could not load TMR model. Check internet connection.')

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'[OK] Model loaded on {device.upper()} ({loaded_model_name})')


def detect_text(text: str):
    '''Chunk-aware AI detection supporting long text and resumes.'''
    if not text or len(text.strip()) < 20:
        return {'error': 'Text too short (minimum 20 chars required)'}

    # Split long text into paragraphs / chunks of max 350 words
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    chunks = []
    curr_chunk = []
    curr_len = 0

    for p in (paragraphs if len(paragraphs) > 1 else [text]):
        words = p.split()
        if curr_len + len(words) > 350:
            if curr_chunk:
                chunks.append(' '.join(curr_chunk))
                curr_chunk = []
                curr_len = 0
        curr_chunk.append(p)
        curr_len += len(words)

    if curr_chunk:
        chunks.append(' '.join(curr_chunk))

    if not chunks:
        chunks = [text[:2000]]

    ai_scores = []
    human_scores = []

    with torch.no_grad():
        for chunk in chunks:
            inputs = tokenizer(
                chunk,
                return_tensors='pt',
                truncation=True,
                max_length=512,
                padding=True
            ).to(device)
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[0]
            human_scores.append(probs[0].item() * 100)
            ai_scores.append(probs[1].item() * 100)

    avg_ai = sum(ai_scores) / len(ai_scores)
    avg_human = sum(human_scores) / len(human_scores)

    ai_prob = round(avg_ai, 1)
    human_prob = round(avg_human, 1)
    verdict = 'AI' if ai_prob >= 50 else 'Human'
    label = 'AI-Generated' if verdict == 'AI' else 'Likely Human'

    return {
        'ai_probability': ai_prob,
        'human_probability': human_prob,
        'verdict': verdict,
        'label': label,
        'model': loaded_model_name,
        'chunks_analyzed': len(chunks)
    }


# Gradio Interface
with gr.Blocks(title='TMR AI Detector Server') as demo:
    gr.Markdown(f'# TMR AI Text Detector API\nPowered by `{loaded_model_name}` on **{device.upper()}**')
    with gr.Row():
        input_box = gr.Textbox(label='Input Text', lines=8, placeholder='Paste text to analyze...')
        output_box = gr.JSON(label='Detection Output')
    btn = gr.Button('Analyze Text', variant='primary')
    btn.click(fn=detect_text, inputs=input_box, outputs=output_box, api_name='predict')

print('\n' + '='*65)
print('  Launching Gradio Public Server...')
print('  Copy the "Running on public URL: https://xxxx.gradio.live" link below')
print('  and paste it into your local ATS Agent .env as COLAB_DETECTOR_URL=')
print('='*65 + '\n')

demo.launch(share=True, show_error=True)
